# TAD Consensus Pipeline — Интерактивный анализ
GM12878 (hg19), Rao et al. 2014

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import yaml

from src.data_prep import load_config, get_matrix
from src.consensus import compute_consensus, CONSENSUS_COLORS
from src.statistics import compute_basic_stats, jaccard_domains
from src.visualization import plot_tad_comparison

cfg = load_config('../config/config.yaml')
print('Конфиг загружен:', cfg['project']['name'])

In [ ]:
# Загрузка результатов
chrom      = 'chr17'
resolution = 50_000

algorithms = ['armatus', 'topdom', 'scktld', 'coitad']
results = {}
for algo in algorithms:
    path = f'../results/tads/{algo}_{chrom}_{resolution}bp.bed'
    try:
        results[algo] = pd.read_csv(
            path, sep='\t', header=None,
            names=['chrom', 'start', 'end']
        )
        print(f'{algo}: {len(results[algo])} TADs')
    except FileNotFoundError:
        print(f'{algo}: файл не найден')
        results[algo] = pd.DataFrame(columns=['chrom', 'start', 'end'])

In [ ]:
# Консенсус
consensus_df = compute_consensus(results, chrom, resolution)
print(f'Консенсусных границ: {len(consensus_df)}')
consensus_df['support'].value_counts().sort_index().plot.bar(
    color=[CONSENSUS_COLORS.get(i, 'gray') for i in [2, 3, 4]],
    title='Распределение поддержки консенсусных границ'
)
plt.show()

In [ ]:
# Базовая статистика
stats_rows = []
for algo, df in results.items():
    s = compute_basic_stats(df, chrom, resolution)
    stats_rows.append({'algorithm': algo, **s})

stats_df = pd.DataFrame(stats_rows)
display(stats_df)

In [ ]:
# Hi-C карта + TAD (только если матрица доступна)
try:
    matrix = get_matrix(cfg, chrom, resolution)
    plot_tad_comparison(
        matrix, results, consensus_df,
        chrom, resolution,
        out_path=f'../results/figures/exploration_{chrom}_{resolution}bp.png',
        cfg=cfg
    )
    from IPython.display import Image
    display(Image(f'../results/figures/exploration_{chrom}_{resolution}bp.png'))
except Exception as e:
    print(f'Матрица недоступна: {e}')